# 策略 × 行业矩阵回测

对比 Top 5 策略在不同行业板块的表现，找出「什么策略适合什么行业」。

**策略池**：reversed_gtja_vwap, gtja_vwap, gtja_momentum, gtja_volume_price, gtja_volatility

**股票池**：default.yaml 中的 100 只 CSI 300 成分股，按行业分为 10 组

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

from src.config.loader import load_config
from src.data.fetcher import fetch_daily
from src.data.storage import save_parquet, load_parquet
from src.data.filters import detect_limit_price, detect_suspension
from src.data.universe import resolve_universe
from src.data import validate_ohlcv
from src.analysis.pool_matrix import run_matrix, pivot_matrix, best_per_pool

print('All imports OK')

All imports OK


## Step 1: 加载数据

In [2]:
cfg = load_config(Path('../configs/default.yaml'))
universe_cfg = cfg.get('universe', {})

STOCKS = resolve_universe(universe_cfg)
START = universe_cfg.get('start_date', '2023-01-01')
END = universe_cfg.get('end_date', '2026-05-23')
RAW_DIR = Path('../data/raw')

print(f'Loading {len(STOCKS)} stocks from {START} to {END}...')

frames = []
for code in STOCKS:
    path = RAW_DIR / f'{code}.parquet'
    if path.exists():
        df = load_parquet(path)
    else:
        df = fetch_daily(code, START, END)
        save_parquet(df, path)
    frames.append(df)

data = pd.concat(frames, ignore_index=True)
data = detect_limit_price(data)
data = detect_suspension(data)
validate_ohlcv(data)

print(f'Data: {len(data)} rows, {data["code"].nunique()} stocks')
print(f'Date range: {data["date"].min().date()} ~ {data["date"].max().date()}')

Loading 100 stocks from 2023-01-01 to 2026-05-23...


Data: 80990 rows, 100 stocks
Date range: 2023-01-03 ~ 2026-05-22


## Step 2: 行业分组

将 100 只股票按行业分为 10 组。

In [3]:
# 行业分组（基于公司主营业务手动分类）
INDUSTRY_GROUPS = {
    '银行': [
        '601939', '601398', '601288', '601988', '600036',
        '601658', '601328', '601998', '601166', '600000',
        '000001', '601818', '002142', '600919',
    ],
    '非银金融': [
        '601318', '601628', '601601', '601319', '601336',  # 保险
        '600030', '300059', '601211', '300033',             # 证券
        '601066', '601688', '601995',
    ],
    '能源资源': [
        '601857', '600938', '600028',  # 石油
        '601088', '601225', '601898', '600188',  # 煤炭
        '603993', '601899', '600111', '601600', '600362',  # 有色
    ],
    '科技半导体': [
        '688981', '688256', '688041', '002371', '688008',
        '603986', '688012', '002415', '002916', '002463',
        '600183',
    ],
    '通信设备': [
        '600941', '601728',  # 运营商
        '300308', '300502', '300394', '000063',  # 光模块/通信设备
    ],
    '消费': [
        '600519', '000858', '600809',  # 白酒
        '000651', '000333', '600690',  # 家电
        '603288', '600887', '002050',  # 食品/其他
    ],
    '医药': [
        '600276', '603259', '300760',
    ],
    '电力公用': [
        '600900', '600930', '003816', '601985', '600025',
    ],
    '新能源': [
        '300750', '002594', '300274',  # 电池/光伏
        '000792', '002460',  # 锂矿
    ],
    '电子制造': [
        '601138', '002475', '002384', '300476', '300433',
        '002938', '300408', '000725', '000338', '600309',
    ],
    '装备制造': [
        '600150', '300124', '600406', '600031', '601100',
        '601766', '302132', '600989', '002714', '601668',
        '601816', '601919', '002352',
    ],
}

# 验证所有股票都被覆盖
all_grouped = set()
for name, codes in INDUSTRY_GROUPS.items():
    all_grouped.update(codes)

all_stocks = set(STOCKS)
missing = all_stocks - all_grouped
extra = all_grouped - all_stocks

print(f'Groups: {len(INDUSTRY_GROUPS)}')
print(f'Total stocks in groups: {len(all_grouped)}')
print(f'Stocks in config: {len(all_stocks)}')
if missing:
    print(f'MISSING from groups: {missing}')
if extra:
    print(f'EXTRA (not in config): {extra}')

for name, codes in INDUSTRY_GROUPS.items():
    print(f'  {name}: {len(codes)} stocks')

Groups: 11
Total stocks in groups: 100
Stocks in config: 100
  银行: 14 stocks
  非银金融: 12 stocks
  能源资源: 12 stocks
  科技半导体: 11 stocks
  通信设备: 6 stocks
  消费: 9 stocks
  医药: 3 stocks
  电力公用: 5 stocks
  新能源: 5 stocks
  电子制造: 10 stocks
  装备制造: 13 stocks


## Step 3: 定义策略

In [4]:
# Top 5 策略（按历史回测表现）
STRATEGY_SPECS = [
    {'name': 'reversed_gtja_vwap'},
    {'name': 'gtja_vwap'},
    {'name': 'gtja_momentum'},
    {'name': 'gtja_volume_price'},
    {'name': 'gtja_volatility'},
]

print('Strategies:')
for s in STRATEGY_SPECS:
    print(f'  - {s["name"]}')

Strategies:
  - reversed_gtja_vwap
  - gtja_vwap
  - gtja_momentum
  - gtja_volume_price
  - gtja_volatility


## Step 4: 运行矩阵回测

每个行业 × 每个策略 = 50 个组合，各自跑完整回测管道。

In [5]:
results = run_matrix(
    pool_groups=INDUSTRY_GROUPS,
    strategy_specs=STRATEGY_SPECS,
    data=data,
    capital=1_000_000,
    max_weight=0.3,
)

print(f'Results: {len(results)} rows ({len(INDUSTRY_GROUPS)} pools × {len(STRATEGY_SPECS)} strategies)')
results

Results: 55 rows (11 pools × 5 strategies)


,strategy,pool,total_return,annual_return,sharpe_ratio,max_drawdown,win_rate,trade_count
0,reversed_gtja_vwap,银行,0.601862,0.156420,8.678190e-01,0.153589,1.000000,7
1,gtja_vwap,银行,0.092414,0.027639,7.584988e-02,0.248255,0.000000,5
2,gtja_momentum,银行,0.416992,0.113496,5.795748e-01,0.202144,0.578512,247
3,gtja_volume_price,银行,0.496687,0.132448,6.770842e-01,0.179544,0.527273,225
4,gtja_volatility,银行,0.262065,0.074430,3.383624e-01,0.198167,0.565217,235
5,reversed_gtja_vwap,非银金融,0.227415,0.065244,2.578888e-01,0.321662,0.472727,113
6,gtja_vwap,非银金融,0.233917,0.066981,2.651605e-01,0.274673,0.556522,235
7,gtja_momentum,非银金融,0.153772,0.045107,1.897487e-01,0.287896,0.457364,263
8,gtja_volume_price,非银金融,-0.116382,-0.037445,-1.474480e-02,0.516327,0.462185,243
9,gtja_volatility,非银金融,-0.056233,-0.017693,-5.377950e-02,0.283963,0.468468,227


## Step 5: Sharpe 矩阵

每个行业-策略组合的 Sharpe Ratio。

In [6]:
sharpe_pivot = pivot_matrix(results, metric='sharpe_ratio')
sharpe_pivot

strategy,gtja_momentum,gtja_volatility,gtja_volume_price,gtja_vwap,reversed_gtja_vwap
pool,,,,,
医药,-0.050885,-0.050885,-0.045633,-0.050885,-6.972214e+16
新能源,0.082504,0.082504,0.118239,0.082504,-6.972214e+16
消费,0.064604,-0.094329,-0.592530,-0.136618,-9.602826e-01
电力公用,0.556895,0.556895,0.556568,0.556895,-6.972214e+16
电子制造,1.034337,0.993840,1.222134,1.006790,1.314991e+00
科技半导体,1.293983,1.382932,0.854424,1.163923,1.275655e+00
能源资源,0.818662,0.710023,0.300083,0.559542,9.891512e-01
装备制造,0.216503,0.377556,0.368989,0.502585,3.972926e-01
通信设备,1.491467,1.394766,1.469226,1.461637,1.436528e+00


In [7]:
# 热力图
fig, ax = plt.subplots(figsize=(12, 7))

im = ax.imshow(sharpe_pivot.values, aspect='auto', cmap='RdYlGn')

ax.set_xticks(range(len(sharpe_pivot.columns)))
ax.set_xticklabels(sharpe_pivot.columns, rotation=45, ha='right')
ax.set_yticks(range(len(sharpe_pivot.index)))
ax.set_yticklabels(sharpe_pivot.index)

# 在每个格子内标注数值
for i in range(len(sharpe_pivot.index)):
    for j in range(len(sharpe_pivot.columns)):
        val = sharpe_pivot.iloc[i, j]
        text = f'{val:.2f}' if not pd.isna(val) else '—'
        # 根据背景亮度选文字颜色
        bg = im.norm(val) if not pd.isna(val) else 0.5
        color = 'white' if bg < 0.4 or bg > 0.6 else 'black'
        ax.text(j, i, text, ha='center', va='center', fontsize=10, color=color)

ax.set_title('Strategy × Industry — Sharpe Ratio', fontsize=14, fontweight='bold')
fig.colorbar(im, ax=ax, shrink=0.8, label='Sharpe')
fig.tight_layout()

output_dir = Path('output')
output_dir.mkdir(exist_ok=True)
fig.savefig(output_dir / 'industry_matrix_sharpe.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: industry_matrix_sharpe.png')

Saved: industry_matrix_sharpe.png


/var/folders/_7/p1ny6h2j1rj72ddzftb81k440000gn/T/ipykernel_24624/2930272053.py:23: UserWarning: Glyph 21307 (\N{CJK UNIFIED IDEOGRAPH-533B}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/var/folders/_7/p1ny6h2j1rj72ddzftb81k440000gn/T/ipykernel_24624/2930272053.py:23: UserWarning: Glyph 33647 (\N{CJK UNIFIED IDEOGRAPH-836F}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/var/folders/_7/p1ny6h2j1rj72ddzftb81k440000gn/T/ipykernel_24624/2930272053.py:23: UserWarning: Glyph 26032 (\N{CJK UNIFIED IDEOGRAPH-65B0}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/var/folders/_7/p1ny6h2j1rj72ddzftb81k440000gn/T/ipykernel_24624/2930272053.py:23: UserWarning: Glyph 33021 (\N{CJK UNIFIED IDEOGRAPH-80FD}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/var/folders/_7/p1ny6h2j1rj72ddzftb81k440000gn/T/ipykernel_24624/2930272053.py:23: UserWarning: Glyph 28304 (\N{CJK UNIFIED IDEOGRAPH-6E90}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/var/folders/_7/p1ny

## Step 6: 总收益矩阵

In [8]:
return_pivot = pivot_matrix(results, metric='total_return')
return_pivot

strategy,gtja_momentum,gtja_volatility,gtja_volume_price,gtja_vwap,reversed_gtja_vwap
pool,,,,,
医药,-0.034431,-0.034431,-0.032433,-0.034431,0.000000
新能源,0.029368,0.029368,0.068993,0.029368,0.000000
消费,0.084846,-0.022497,-0.292653,-0.032704,-0.421232
电力公用,0.434158,0.434158,0.433627,0.434158,0.000000
电子制造,1.486067,1.423573,1.911078,1.297374,1.603145
科技半导体,1.710185,1.977649,1.167207,1.465974,1.868790
能源资源,0.863331,0.755393,0.272915,0.484530,0.969744
装备制造,0.168824,0.382805,0.337998,0.474638,0.292874
通信设备,2.129460,2.089356,2.170010,1.999011,0.802201


In [9]:
fig, ax = plt.subplots(figsize=(12, 7))

im = ax.imshow(return_pivot.values, aspect='auto', cmap='RdYlGn')

ax.set_xticks(range(len(return_pivot.columns)))
ax.set_xticklabels(return_pivot.columns, rotation=45, ha='right')
ax.set_yticks(range(len(return_pivot.index)))
ax.set_yticklabels(return_pivot.index)

for i in range(len(return_pivot.index)):
    for j in range(len(return_pivot.columns)):
        val = return_pivot.iloc[i, j]
        text = f'{val:.1%}' if not pd.isna(val) else '—'
        bg = im.norm(val) if not pd.isna(val) else 0.5
        color = 'white' if bg < 0.4 or bg > 0.6 else 'black'
        ax.text(j, i, text, ha='center', va='center', fontsize=10, color=color)

ax.set_title('Strategy × Industry — Total Return', fontsize=14, fontweight='bold')
fig.colorbar(im, ax=ax, shrink=0.8, label='Return')
fig.tight_layout()
fig.savefig(output_dir / 'industry_matrix_return.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: industry_matrix_return.png')

/var/folders/_7/p1ny6h2j1rj72ddzftb81k440000gn/T/ipykernel_24624/3722037186.py:20: UserWarning: Glyph 21307 (\N{CJK UNIFIED IDEOGRAPH-533B}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/var/folders/_7/p1ny6h2j1rj72ddzftb81k440000gn/T/ipykernel_24624/3722037186.py:20: UserWarning: Glyph 33647 (\N{CJK UNIFIED IDEOGRAPH-836F}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/var/folders/_7/p1ny6h2j1rj72ddzftb81k440000gn/T/ipykernel_24624/3722037186.py:20: UserWarning: Glyph 26032 (\N{CJK UNIFIED IDEOGRAPH-65B0}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/var/folders/_7/p1ny6h2j1rj72ddzftb81k440000gn/T/ipykernel_24624/3722037186.py:20: UserWarning: Glyph 33021 (\N{CJK UNIFIED IDEOGRAPH-80FD}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/var/folders/_7/p1ny6h2j1rj72ddzftb81k440000gn/T/ipykernel_24624/3722037186.py:20: UserWarning: Glyph 28304 (\N{CJK UNIFIED IDEOGRAPH-6E90}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/var/folders/_7/p1ny

Saved: industry_matrix_return.png


## Step 7: 最大回撤矩阵

In [10]:
dd_pivot = pivot_matrix(results, metric='max_drawdown')

fig, ax = plt.subplots(figsize=(12, 7))

im = ax.imshow(dd_pivot.values, aspect='auto', cmap='RdYlGn_r')  # 反转：回撤越小越好

ax.set_xticks(range(len(dd_pivot.columns)))
ax.set_xticklabels(dd_pivot.columns, rotation=45, ha='right')
ax.set_yticks(range(len(dd_pivot.index)))
ax.set_yticklabels(dd_pivot.index)

for i in range(len(dd_pivot.index)):
    for j in range(len(dd_pivot.columns)):
        val = dd_pivot.iloc[i, j]
        text = f'{val:.1%}' if not pd.isna(val) else '—'
        bg = im.norm(val) if not pd.isna(val) else 0.5
        color = 'white' if bg < 0.4 or bg > 0.6 else 'black'
        ax.text(j, i, text, ha='center', va='center', fontsize=10, color=color)

ax.set_title('Strategy × Industry — Max Drawdown (lower is better)', fontsize=14, fontweight='bold')
fig.colorbar(im, ax=ax, shrink=0.8, label='Max Drawdown')
fig.tight_layout()
fig.savefig(output_dir / 'industry_matrix_drawdown.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: industry_matrix_drawdown.png')

Saved: industry_matrix_drawdown.png


/var/folders/_7/p1ny6h2j1rj72ddzftb81k440000gn/T/ipykernel_24624/3097194474.py:22: UserWarning: Glyph 21307 (\N{CJK UNIFIED IDEOGRAPH-533B}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/var/folders/_7/p1ny6h2j1rj72ddzftb81k440000gn/T/ipykernel_24624/3097194474.py:22: UserWarning: Glyph 33647 (\N{CJK UNIFIED IDEOGRAPH-836F}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/var/folders/_7/p1ny6h2j1rj72ddzftb81k440000gn/T/ipykernel_24624/3097194474.py:22: UserWarning: Glyph 26032 (\N{CJK UNIFIED IDEOGRAPH-65B0}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/var/folders/_7/p1ny6h2j1rj72ddzftb81k440000gn/T/ipykernel_24624/3097194474.py:22: UserWarning: Glyph 33021 (\N{CJK UNIFIED IDEOGRAPH-80FD}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/var/folders/_7/p1ny6h2j1rj72ddzftb81k440000gn/T/ipykernel_24624/3097194474.py:22: UserWarning: Glyph 28304 (\N{CJK UNIFIED IDEOGRAPH-6E90}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/var/folders/_7/p1ny

## Step 8: 每个行业最优策略

In [11]:
print('=== Best by Sharpe ===')
best_sharpe = best_per_pool(results, metric='sharpe_ratio')
best_sharpe

=== Best by Sharpe ===


,pool,strategy,sharpe_ratio
0,医药,gtja_volume_price,-0.045633
1,新能源,gtja_volume_price,0.118239
2,消费,gtja_momentum,0.064604
3,电力公用,gtja_vwap,0.556895
4,电子制造,reversed_gtja_vwap,1.314991
5,科技半导体,gtja_volatility,1.382932
6,能源资源,reversed_gtja_vwap,0.989151
7,装备制造,gtja_vwap,0.502585
8,通信设备,gtja_momentum,1.491467
9,银行,reversed_gtja_vwap,0.867819


In [12]:
print('=== Best by Total Return ===')
best_return = best_per_pool(results, metric='total_return')
best_return

=== Best by Total Return ===


,pool,strategy,total_return
0,医药,reversed_gtja_vwap,0.000000
1,新能源,gtja_volume_price,0.068993
2,消费,gtja_momentum,0.084846
3,电力公用,gtja_vwap,0.434158
4,电子制造,gtja_volume_price,1.911078
5,科技半导体,gtja_volatility,1.977649
6,能源资源,reversed_gtja_vwap,0.969744
7,装备制造,gtja_vwap,0.474638
8,通信设备,gtja_volume_price,2.170010
9,银行,reversed_gtja_vwap,0.601862


In [13]:
print('=== Best by Max Drawdown (lowest) ===')
best_dd = best_per_pool(results, metric='max_drawdown')
best_dd

=== Best by Max Drawdown (lowest) ===


,pool,strategy,max_drawdown
0,医药,gtja_volume_price,0.280087
1,新能源,gtja_vwap,0.484240
2,消费,reversed_gtja_vwap,0.436238
3,电力公用,gtja_vwap,0.255059
4,电子制造,gtja_momentum,0.356405
5,科技半导体,gtja_volume_price,0.363214
6,能源资源,gtja_volume_price,0.322285
7,装备制造,gtja_momentum,0.449466
8,通信设备,gtja_volatility,0.272727
9,银行,gtja_vwap,0.248255


## Step 9: 策略排名汇总

统计每个策略在所有行业中的平均排名。（排名 1 = 最优）

In [14]:
# 每个行业内部按 Sharpe 排名
results['sharpe_rank'] = results.groupby('pool')['sharpe_ratio'].rank(ascending=False)

ranking = results.groupby('strategy').agg(
    avg_rank=('sharpe_rank', 'mean'),
    win_count=('sharpe_rank', lambda x: (x == 1).sum()),
    avg_sharpe=('sharpe_ratio', 'mean'),
    avg_return=('total_return', 'mean'),
).sort_values('avg_rank')

ranking

,avg_rank,win_count,avg_sharpe,avg_return
strategy,,,,
gtja_momentum,2.545455,2,5.706722e-01,0.676597
gtja_vwap,3.000000,2,4.988532e-01,0.585841
gtja_volume_price,3.090909,2,4.467128e-01,0.583368
reversed_gtja_vwap,3.090909,3,-1.901513e+16,0.540436
gtja_volatility,3.272727,1,5.125350e-01,0.658291


## Step 10: 分析总结

（运行后填写观察）

- 哪些策略在哪些行业表现最好？
- 是否存在「万能策略」在所有行业都有效？
- 行业因素对策略表现的影响有多大？